# 1. Library calling

In [1]:
#pip install selenium
#pip install webdriver_manager
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib.parse import unquote
from selenium.webdriver.support.ui import Select
import pandas as pd
import datetime 

import warnings

# Ignore all warnings (not recommended in general)
warnings.filterwarnings("ignore")

# 2. Defining the Product Information and Location

In [3]:
SummaryFolder=r'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects'
summaryFile='Scraping_List.txt'
st=pd.read_csv(SummaryFolder+'\\'+summaryFile)
print(st)
#Define Product to extract
search_text = st['Product Name'][73]
print(search_text)
Source="Amazon"
OFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Outputs'
IFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Inputs'

                           Product Name
0                         Ignition Coil
1               Windshield Washer Pumps
2                 coupler trailer locks
3   Adjustable Trailer Hitch Ball Mount
4                        Vacuum Cleaner
..                                  ...
69                  Brake Spring Pliers
70               swing away hitch mount
71                      Garage Products
72                        Oxygen Sensor
73             Brand Specific O2 Sensor

[74 rows x 1 columns]
Brand Specific O2 Sensor


# 3. Setting Webdriver and Website Specific Information

In [4]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
driver.get('https://www.amazon.com/')

### Please update capatche.

In [5]:
location = driver.find_element(By.ID, 'nav-global-location-popover-link')

location.click()
sleep(1)
pin = driver.find_element(By.ID, 'GLUXZipUpdateInput')
pinnumber='94203'
pin.send_keys(pinnumber)

apply=driver.find_element(By.CSS_SELECTOR, 'span.a-button-inner[data-action="GLUXPostalUpdateAction"]')
apply.click()


In [5]:
#driver.maximize_window()

In [7]:
dropdown = driver.find_element(By.ID,'searchDropdownBox')

# Create a Select object
select = Select(dropdown)

# Select an option by visible text
select.select_by_visible_text('Automotive Parts & Accessories')

In [6]:
# Find the search bar by its ID
try:
    search_bar = driver.find_element(By.ID, 'nav-bb-search')
except:
    search_bar=driver.find_element(By.ID, "twotabsearchtextbox")

# Enter the search text

search_bar.send_keys(search_text)

# Submit the form to perform the search
search_bar.submit()

# 4. Defining the Dataframe and Extracting the data into the Dataframe

In [57]:
Cols=["Links","Name"]
df = pd.DataFrame(columns=Cols)
count=0

In [10]:
Linklist=driver.find_elements(By.CSS_SELECTOR,'[class="a-link-normal aok-block"]')
Namelist=driver.find_elements(By.CSS_SELECTOR,'[class="_cDEzb_p13n-sc-css-line-clamp-3_g3dy1"]')

for link,name in zip(Linklist,Namelist):
    df.loc[count,"Links"]=link.get_attribute("href").split("/ref")[0]
    df.loc[count,"Name"]=name.text
    count=count+1

In [58]:
df

,Links,Name


In [ ]:
[1].find_element(By.CSS_SELECTOR,'[class="a-price a-text-price"]').text

In [48]:
driver.find_elements(By.CSS_SELECTOR,'[data-cy="price-recipe"]')[1].text.split('\n')[-1]

'$169.99'

In [68]:
for i in range(20):
    elemento = driver.find_element(By.CLASS_NAME,"a-spacing-small")
    # Print the text content of the element
    #print("Element Text:", element.text)
    PageNumber=elemento.text.split('result')[0]
    PageRange= PageNumber.split(" ")[0]
    TotalPage=PageNumber.split(" ")[2]
    max= PageRange.split("-")[1]
    print(max, TotalPage)
    sleep(5)
    elements=driver.find_elements(By.CLASS_NAME,'s-title-instructions-style')
    ratings=driver.find_elements(By.CSS_SELECTOR,'[data-cy="reviews-block"]')
    ScreenPrices=driver.find_elements(By.CLASS_NAME,'a-price')
    Listprices=driver.find_elements(By.CSS_SELECTOR,'[data-cy="price-recipe"]')
    for element,rating,sp,lp in zip( elements, ratings,ScreenPrices,Listprices):
        try:
            df.loc[count,'Brand']=element.find_element(By.CSS_SELECTOR,'[class="a-size-base-plus a-color-base"]').text
        except:
            pass
        df.loc[count,'Name']=element.find_element(By.CLASS_NAME,"a-text-normal").text
        df.loc[count,'Links']=element.find_element(By.CLASS_NAME,"a-text-normal").get_attribute('href')
        df.loc[count,'#Rating']=int(rating.text.split("\n")[0].replace(",",""))

        try:
            df.loc[count,'SellingPrice']=float(sp.text.replace("\n",".").replace("$","").replace(",",""))
        except:
            pass
        try:
            df.loc[count,'ListPrice']=float(lp.text.split('\n')[-1])
        except:
            pass

        try:
            df.loc[count,'Sponsored?']=element.find_element(By.CSS_SELECTOR,'[class="a-color-secondary"]').text
        except:
            pass
        count=count+1
    if max!= TotalPage:
        next_button = driver.find_element(By.CLASS_NAME,'s-pagination-next')
        next_button.click()
        i=i+1
    else:
        break

24 839
24 839
48 839
72 839
96 839


KeyboardInterrupt: 

In [69]:
df

,Links,Name,#Rating,SellingPrice,ListPrice,Sponsored?
0,https://www.amazon.com/Bosch-Automotive-17025-...,17025 Oxygen Sensor,2433.0,75.60,60.0,NaN
1,https://www.amazon.com/Bosch-17321-Original-Eq...,17321 Premium Original Equipment Oxygen Sensor...,442.0,66.64,NaN,NaN
2,https://www.amazon.com/Bosch-15717-Original-Eq...,15717 Premium Original Equipment Oxygen Sensor...,1603.0,76.72,NaN,NaN
3,https://www.amazon.com/Bosch-17098-Oxygen-Orig...,17098 Premium Original Equipment Oxygen Sensor...,350.0,29.05,NaN,NaN
4,https://www.amazon.com/Bosch-13474-Cadillac-Ch...,13474 Premium OE Fitment Oxygen Sensor - Compa...,916.0,35.22,NaN,NaN
...,...,...,...,...,...,...
1431,https://www.amazon.com/ChawYI-Downstream-Compa...,12701634 Downstream or Upstream Oxygen O2 Sens...,1.0,21.93,NaN,NaN
1432,https://www.amazon.com/DOSKJOK-234-4587-250-24...,234-4587 O2 Oxygen Sensor 250-24253 Compatible...,32.0,29.99,99.0,NaN
1433,https://www.amazon.com/ACDelco-213-1702-Origin...,GM Genuine Parts 213-1702 Heated Oxygen Sensor,528.0,69.99,8.0,NaN
1434,https://www.amazon.com/ACAUTO-Upstream-Downstr...,15717 Oxygen Sensor Upstream Downstream O2 Sen...,73.0,45.93,99.0,NaN


# 5. Post Processing Data and Exporting

In [8]:
items_to_remove = ["javascript:void(0)"]
df = df[~df['Links'].isin(items_to_remove)]
df

,Links,Name,Sponsored?
0,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,"KAC Swing Away Hitch Adapter, 90-Degree Swing ...",Sponsored
1,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,"Swing Away Hitch Adapter, 90-Degree Swing Arm,...",Sponsored
2,https://www.amazon.com/Extension-Adjustable-12...,KUAT Pivot V2 Swing Away Bike Rack Extension |...,NaN
3,https://www.amazon.com/KAC-90-Degree-Capacity-...,"KAC Swing Away Hitch Adapter, 90-Degree Swing ...",NaN
4,https://www.amazon.com/YAKIMA-BackSwing-Swing-...,"YAKIMA, BackSwing, Swing-Away Bike Rack Adapte...",NaN
...,...,...,...
184,https://www.amazon.com/DRAW-TITE-75758-Draw-Ti...,"Draw-Tite 75758 Max-Frame Receiver , Black",NaN
185,https://www.amazon.com/Draw-Tite-Trailer-Recei...,"Draw-Tite 41116 Class 3 Trailer Hitch, 2 Inch ...",NaN
186,https://www.amazon.com/Draw-Tite-75119-Max-Fra...,Draw-Tite 75119 Max-Frame Class III Round Rece...,NaN
187,https://www.amazon.com/Draw-Tite-75662-Hitch-R...,"Draw-Tite 75662 Hitch for RAM 1500 '09 , Black",NaN


In [9]:
df['Links1']=df['Links'].str.replace('%',"&").str.replace('&2F','/')

In [10]:
print(len(df))
df

189


,Links,Name,Sponsored?,Links1
0,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,"KAC Swing Away Hitch Adapter, 90-Degree Swing ...",Sponsored,https://www.amazon.com/sspa/click?ie=UTF8&spc=...
1,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,"Swing Away Hitch Adapter, 90-Degree Swing Arm,...",Sponsored,https://www.amazon.com/sspa/click?ie=UTF8&spc=...
2,https://www.amazon.com/Extension-Adjustable-12...,KUAT Pivot V2 Swing Away Bike Rack Extension |...,NaN,https://www.amazon.com/Extension-Adjustable-12...
3,https://www.amazon.com/KAC-90-Degree-Capacity-...,"KAC Swing Away Hitch Adapter, 90-Degree Swing ...",NaN,https://www.amazon.com/KAC-90-Degree-Capacity-...
4,https://www.amazon.com/YAKIMA-BackSwing-Swing-...,"YAKIMA, BackSwing, Swing-Away Bike Rack Adapte...",NaN,https://www.amazon.com/YAKIMA-BackSwing-Swing-...
...,...,...,...,...
184,https://www.amazon.com/DRAW-TITE-75758-Draw-Ti...,"Draw-Tite 75758 Max-Frame Receiver , Black",NaN,https://www.amazon.com/DRAW-TITE-75758-Draw-Ti...
185,https://www.amazon.com/Draw-Tite-Trailer-Recei...,"Draw-Tite 41116 Class 3 Trailer Hitch, 2 Inch ...",NaN,https://www.amazon.com/Draw-Tite-Trailer-Recei...
186,https://www.amazon.com/Draw-Tite-75119-Max-Fra...,Draw-Tite 75119 Max-Frame Class III Round Rece...,NaN,https://www.amazon.com/Draw-Tite-75119-Max-Fra...
187,https://www.amazon.com/Draw-Tite-75662-Hitch-R...,"Draw-Tite 75662 Hitch for RAM 1500 '09 , Black",NaN,https://www.amazon.com/Draw-Tite-75662-Hitch-R...


In [11]:
df['ASINs'] = df['Links1'].str.split('''/dp/''',expand=True)[1].str.split('''/ref''',expand=True)[0]
del df["Links1"]
prefix="https://www.amazon.com/dp/"
df['Links']=prefix+df["ASINs"]


In [12]:
df=df.drop_duplicates(subset=['ASINs'])
df

,Links,Name,Sponsored?,ASINs
0,https://www.amazon.com/dp/B0B2T3F3HM,"KAC Swing Away Hitch Adapter, 90-Degree Swing ...",Sponsored,B0B2T3F3HM
1,https://www.amazon.com/dp/B0DDKBLY69,"Swing Away Hitch Adapter, 90-Degree Swing Arm,...",Sponsored,B0DDKBLY69
2,https://www.amazon.com/dp/B07YVGYPHS,KUAT Pivot V2 Swing Away Bike Rack Extension |...,NaN,B07YVGYPHS
4,https://www.amazon.com/dp/B077BX789P,"YAKIMA, BackSwing, Swing-Away Bike Rack Adapte...",NaN,B077BX789P
5,https://www.amazon.com/dp/B088K5PLCZ,"Swing Away Hitch Adapter by Saris | 2"" Receive...",NaN,B088K5PLCZ
...,...,...,...,...
183,https://www.amazon.com/dp/B09Y69JQLG,"Draw-Tite 76543 Class 3 Trailer Hitch, 2-Inch ...",NaN,B09Y69JQLG
184,https://www.amazon.com/dp/B00CMKWKFE,"Draw-Tite 75758 Max-Frame Receiver , Black",NaN,B00CMKWKFE
185,https://www.amazon.com/dp/B000E283U8,"Draw-Tite 41116 Class 3 Trailer Hitch, 2 Inch ...",NaN,B000E283U8
186,https://www.amazon.com/dp/B000O5I6D4,Draw-Tite 75119 Max-Frame Class III Round Rece...,NaN,B000O5I6D4


In [70]:
df.to_excel(IFolder+'\\'+f'{Source}ProductLinks_'+search_text+'.xlsx')#

# 99. Archived Codes

In [ ]:
sortbutton=driver.find_element(By.XPATH,"//span[@id='a-autoid-0-announce']")
#sortbutton=driver.find_element(By.CLASS_NAME,"a-button-text a-declarative")
sortbutton.click()
BSbutton=driver.find_element(By.XPATH,"//a[contains(@class, 'a-dropdown-link') and text()='Best Sellers']")
BSbutton.click()